# Parsing St John's ambulance first aid reference doc

In [1]:
import pdfplumber

PDF_PATH = "st-johns-first-aid-reference-guide.pdf"

# Pages to SKIP (0-indexed): cover, contents, being a first aider section
# Corresponds to pages 1–14 in the printed guide
SKIP_PAGES = set(range(0, 14))  # pages 0–13 (0-indexed) = printed pages 1–14

In [2]:
# ── Cell 2: Quick inspection ───────────────────────────────────────────────
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    
    # Preview a known-good page to check extraction quality
    # Page 15 (0-indexed: 14) = start of "Assessing a casualty"
    sample = pdf.pages[14].extract_text()
    print("\n── Sample (page 15) ──────────────────────────────")
    print(sample)

Total pages: 72

── Sample (page 15) ──────────────────────────────
First aid reference guide 15
Assessing a casualty
When you encounter a casualty, you have to assess them so you can deliver
effective, safe and prompt first aid. There are two procedures you must follow to
help you determine what your response should be:
h The first process is called the primary survey which is an initial assessment to
identify and deal with life threatening injuries or conditions
h This can be followed by the secondary survey which is a detailed and
methodical examination of the casualty to look for injuries that may not be
apparent.
Primary survey
The primary survey is often referred to as DRABC. DRABC stands for the five main
parts of the primary survey and is a good way to remember the procedure:
h Danger
h Response
h Airway
h Breathing
h Circulation.
You should apply DRABC to all first aid incidents, however small.
Danger
Before approaching any casualty, first, ensure that there is
no danger to yo

In [3]:
# ── Cell 3: Extract all clinical pages ────────────────────────────────────
pages_text = {}  # {page_num_1indexed: raw_text}

with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        if i in SKIP_PAGES:
            continue
        
        text = page.extract_text()
        
        if text and text.strip():
            pages_text[i + 1] = text  # store as 1-indexed for readability

print(f"Extracted {len(pages_text)} pages of clinical content")

Extracted 58 pages of clinical content


In [4]:
# ── Cell 4: Browse the raw output page by page ────────────────────────────
# Adjust PAGE to inspect different pages
PAGE = 23  # try 15, 23, 39, 55, 67 etc — major section starts

print(f"── Page {PAGE} ──────────────────────────────────────")
print(pages_text.get(PAGE, "Page not found or skipped"))

── Page 23 ──────────────────────────────────────
First aid reference guide 23
Unresponsive not breathing casualty
Cardiopulmonary resuscitation (CPR)
Chest compressions and rescue breaths performed together are called
cardiopulmonary resuscitation (CPR). CPR is an important part of the chain of
survival for a casualty whose heart has stopped. The quality of CPR is dependent on
the correct depth and rate.
Chain of survival
h Recognising cardiac arrest
h Early request for help to the emergency services
h Early basic life support – CPR
h Early defibrillation
h Early advanced life support.
To provide the last two links in the chain you need to get help as soon as possible.
Defibrillation is known to greatly improve the outcome for some casualties whose
hearts have stopped. If a defibrillator is available, get it when calling the emergency
services: a defibrillator should not be used on a baby.
In the first few minutes after the heart has stopped, it is common to see agonal
breathing – sho

In [5]:
# ── Cell 5: Quick look at all page openings ───────────────────────────────
# Useful for spotting where sections start and where parsing breaks down
for page_num, text in sorted(pages_text.items()):
    first_line = text.strip().split('\n')[0] if text.strip() else "(empty)"
    print(f"  p{page_num:>3}: {first_line[:80]}")

  p 15: First aid reference guide 15
  p 16: 16 First aid reference guide
  p 17: First aid reference guide 17
  p 18: 18 First aid reference guide
  p 19: First aid reference guide 19
  p 20: 20 First aid reference guide
  p 21: First aid reference guide 21
  p 22: 22 First aid reference guide
  p 23: First aid reference guide 23
  p 24: 24 First aid reference guide
  p 25: First aid reference guide 25
  p 26: 26 First aid reference guide
  p 27: First aid reference guide 27
  p 28: 28 First aid reference guide
  p 29: First aid reference guide 29
  p 30: 30 First aid reference guide
  p 31: First aid reference guide 31
  p 32: 32 First aid reference guide
  p 33: First aid reference guide 33
  p 34: 34 First aid reference guide
  p 35: First aid reference guide 35
  p 36: 36 First aid reference guide
  p 37: First aid reference guide 37
  p 38: 38 First aid reference guide
  p 39: First aid reference guide 39
  p 40: 40 First aid reference guide
  p 41: First aid reference guide 41
 

In [6]:
# ── Cell 6: Clean up extraction artefacts ─────────────────────────────────
import re

def clean_page_text(text: str) -> str:
    # Remove running header/footer: "First aid reference guide <n>"
    text = re.sub(r'First aid reference guide\s+\d+\s*\n?', '', text)
    
    # Replace the `h` bullet artefact with a clean dash
    # Only replace `h` that appears at the start of a line (after whitespace)
    text = re.sub(r'^\s*h\s+', '- ', text, flags=re.MULTILINE)
    
    # Collapse 3+ blank lines to 2 (keeps section spacing without excess gaps)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Strip leading/trailing whitespace per page
    return text.strip()


# Apply to all extracted pages
pages_clean = {
    page_num: clean_page_text(text)
    for page_num, text in pages_text.items()
}

# Sanity check — view the same sample page after cleaning
print(pages_clean[15])

Assessing a casualty
When you encounter a casualty, you have to assess them so you can deliver
effective, safe and prompt first aid. There are two procedures you must follow to
help you determine what your response should be:
- The first process is called the primary survey which is an initial assessment to
identify and deal with life threatening injuries or conditions
- This can be followed by the secondary survey which is a detailed and
methodical examination of the casualty to look for injuries that may not be
apparent.
Primary survey
The primary survey is often referred to as DRABC. DRABC stands for the five main
parts of the primary survey and is a good way to remember the procedure:
- Danger
- Response
- Airway
- Breathing
- Circulation.
You should apply DRABC to all first aid incidents, however small.
Danger
Before approaching any casualty, first, ensure that there is
no danger to you. If possible, move any potential sources of
danger so that you can carry out first aid without 

In [7]:
# ── Cell 7: Spot-check a few more pages ───────────────────────────────────
for p in [23, 39, 55, 67]:  # CPR, Choking, Burns, Fractures
    print(f"\n{'─'*60}")
    print(f"PAGE {p}")
    print('─'*60)
    print(pages_clean.get(p, "not found")[:600])
    print("...")


────────────────────────────────────────────────────────────
PAGE 23
────────────────────────────────────────────────────────────
Unresponsive not breathing casualty
Cardiopulmonary resuscitation (CPR)
Chest compressions and rescue breaths performed together are called
cardiopulmonary resuscitation (CPR). CPR is an important part of the chain of
survival for a casualty whose heart has stopped. The quality of CPR is dependent on
the correct depth and rate.
Chain of survival
- Recognising cardiac arrest
- Early request for help to the emergency services
- Early basic life support – CPR
- Early defibrillation
- Early advanced life support.
To provide the last two links in the chain you need to get help as soon as possible.

...

────────────────────────────────────────────────────────────
PAGE 39
────────────────────────────────────────────────────────────
Airway and breathing problems
Choking
Food or other objects stuck in the mouth or throat can cause choking. If the object
isn’t clear

In [8]:
# ── Cell 8: Heading detector ──────────────────────────────────────────────
# Strategy: headings are short lines, title-cased or all-caps, not starting
# with a dash (cleaned bullet), not ending with punctuation like . or ,
# We'll print proposed boundaries for manual review before committing.

def is_likely_heading(line: str) -> bool:
    line = line.strip()
    
    if not line:
        return False
    
    # Must be reasonably short — headings are never a full sentence
    if len(line) > 60:
        return False
    
    # Skip cleaned bullet points
    if line.startswith('-'):
        return False
    
    # Skip lines ending with sentence punctuation (they're body text)
    if line.endswith(('.', ',', ':')):
        return False
    
    # Skip lines that are purely numeric (stray page numbers etc)
    if line.isdigit():
        return False
    
    # Must start with a capital letter
    if not line[0].isupper():
        return False
    
    # Skip very common non-heading patterns
    skip_phrases = ['aims -', 'note:', 'if ', 'do not', 'always', 'never']
    if any(line.lower().startswith(p) for p in skip_phrases):
        return False
    
    return True

In [9]:
# ── Cell 9: Stitch pages into one continuous string, tracking page numbers ─
# We'll keep track of where each page starts so we can record page_range
# in metadata later.
 
full_text_with_markers = ""
page_start_positions = {}  # {page_num: char_offset}
 
for page_num in sorted(pages_clean.keys()):
    page_start_positions[page_num] = len(full_text_with_markers)
    full_text_with_markers += pages_clean[page_num] + "\n\n"
 
print(f"Total characters in clinical text: {len(full_text_with_markers):,}")
 

Total characters in clinical text: 98,072


In [10]:
# ── Cell 10: Detect candidate headings and proposed boundaries ─────────────
 
lines = full_text_with_markers.split('\n')
proposed_boundaries = []  # list of (line_index, heading_text)
 
for i, line in enumerate(lines):
    if is_likely_heading(line):
        proposed_boundaries.append((i, line.strip()))
 
print(f"Found {len(proposed_boundaries)} candidate headings:\n")
for idx, (line_idx, heading) in enumerate(proposed_boundaries):
    print(f"  [{idx:>3}] line {line_idx:>4}: {heading}")

Found 273 candidate headings:

  [  0] line    0: Assessing a casualty
  [  1] line    9: Primary survey
  [  2] line   18: Danger
  [  3] line   19: Before approaching any casualty, first, ensure that there is
  [  4] line   26: Response
  [  5] line   39: Airway
  [  6] line   43: Baby
  [  7] line   49: Child and adult
  [  8] line   51: Gently tilt their head back so their mouth opens (don’t
  [  9] line   57: Breathing
  [ 10] line   58: Once you have opened the airway, check whether the casualty
  [ 11] line   64: In the first few minutes after the heart has stopped it is
  [ 12] line   73: Circulation
  [ 13] line   78: Oxygen flow chart
  [ 14] line   82: No
  [ 15] line   83: Assess the casualty. Does the Treat the casualty and diall
  [ 16] line   86: No
  [ 17] line   88: Is the casualty breathing any serious injuries before
  [ 18] line   94: No
  [ 19] line   99: Baby or child Adult
  [ 20] line  100: Give five initial rescue breaths. Perform CPR. If they have
  [ 21] line

In [11]:
# Cell 11: Define the final chunk boundary list
# Each entry: (line_index, condition_name, category, severity)

CHUNK_BOUNDARIES = [
    # ── Assessment ─────────────────────────────────────────────────────────
    (0,    "Assessing a casualty — Primary Survey",   "Assessment",            "core"),
    (110,  "Secondary Survey",                         "Assessment",            "core"),
    (126,  "Head-to-toe Survey",                       "Assessment",            "core"),
    (174,  "Monitoring a Casualty",                    "Assessment",            "core"),

    # ── Unresponsive breathing ─────────────────────────────────────────────
    (201,  "Recovery Position",                        "Unresponsive Casualty", "life-threatening"),

    # ── CPR & AED ──────────────────────────────────────────────────────────
    (233,  "CPR — Adult, Child and Baby",              "Unresponsive Casualty", "life-threatening"),
    (426,  "Automated External Defibrillation (AED)",  "Unresponsive Casualty", "life-threatening"),
    (592,  "CPR after Drowning",                       "Unresponsive Casualty", "life-threatening"),

    # ── Response problems ──────────────────────────────────────────────────
    (612,  "Seizures",                                 "Response Problems",     "serious"),
    (662,  "Febrile Convulsions",                      "Response Problems",     "serious"),
    (693,  "Stroke",                                   "Response Problems",     "life-threatening"),
    (733,  "Head Injuries",                            "Response Problems",     "serious"),
    (782,  "Diabetes — Hypo and Hyperglycaemia",       "Response Problems",     "serious"),

    # ── Airway & Breathing ─────────────────────────────────────────────────
    (835,  "Choking",                                  "Airway & Breathing",    "life-threatening"),
    (889,  "Asthma",                                   "Airway & Breathing",    "serious"),
    (952,  "Allergic Reactions and Anaphylaxis",       "Airway & Breathing",    "life-threatening"),
    (998,  "Auto-injector",                            "Airway & Breathing",    "life-threatening"),
    (1042, "Hyperventilation",                         "Airway & Breathing",    "minor"),

    # ── Circulation ────────────────────────────────────────────────────────
    (1061, "Heart Attack",                             "Circulation",           "life-threatening"),
    (1105, "Angina",                                   "Circulation",           "serious"),
    (1117, "Shock",                                    "Circulation",           "life-threatening"),
    (1155, "Fainting",                                 "Circulation",           "minor"),

    # ── Bleeding ───────────────────────────────────────────────────────────
    (1181, "Bleeding — Minor, Nosebleed, Bruising",   "Bleeding",              "minor"),
    (1272, "Severe Bleeding",                          "Bleeding",              "life-threatening"),
    (1293, "Embedded Objects in Wounds",               "Bleeding",              "serious"),
    (1311, "Penetrating Chest Wound",                  "Bleeding",              "life-threatening"),
    (1346, "Catastrophic Bleeding and Tourniquets",    "Bleeding",              "life-threatening"),
    (1370, "Amputation",                               "Bleeding",              "life-threatening"),

    # ── Burns, Poisons, Foreign Objects ───────────────────────────────────
    (1435, "Burns",                                    "Burns & Poisons",       "serious"),
    (1487, "Chemical Burns",                           "Burns & Poisons",       "serious"),
    (1531, "Poisons — Swallowed and Skin Contact",    "Burns & Poisons",       "serious"),
    (1576, "Poisonous Gases",                          "Burns & Poisons",       "serious"),
    (1604, "Foreign Objects — Eye, Ear, Nose",        "Burns & Poisons",       "minor"),

    # ── Heat & Cold ────────────────────────────────────────────────────────
    (1642, "Hypothermia",                              "Heat & Cold",           "life-threatening"),
    (1682, "Heat Exhaustion",                          "Heat & Cold",           "serious"),
    (1704, "Heatstroke",                               "Heat & Cold",           "life-threatening"),
    (1726, "Sunburn",                                  "Heat & Cold",           "minor"),

    # ── Other Conditions ───────────────────────────────────────────────────
    (1745, "Meningitis",                               "Other Conditions",      "life-threatening"),
    (1767, "Sepsis",                                   "Other Conditions",      "life-threatening"),
    (1796, "Bites and Stings",                         "Other Conditions",      "minor"),
    (1825, "Tick Bite",                                "Other Conditions",      "minor"),

    # ── Bone, Muscle & Joint ───────────────────────────────────────────────
    (1838, "Sprains and Strains",                      "Bone & Muscle",         "minor"),
    (1859, "Dislocations",                             "Bone & Muscle",         "serious"),
    (1876, "Fractures",                                "Bone & Muscle",         "serious"),
    (1912, "Spinal Injuries",                          "Bone & Muscle",         "life-threatening"),
    (1967, "Crush Injuries",                           "Bone & Muscle",         "serious"),
]

print(f"Total chunks defined: {len(CHUNK_BOUNDARIES)}")

Total chunks defined: 46


In [12]:
# ── Cell 12: Split text into chunks using boundary list ───────────────────

lines = full_text_with_markers.split('\n')

chunks = []

for i, (line_idx, condition, category, severity) in enumerate(CHUNK_BOUNDARIES):
    
    # End of this chunk = start of next boundary, or end of document
    if i + 1 < len(CHUNK_BOUNDARIES):
        end_line = CHUNK_BOUNDARIES[i + 1][0]
    else:
        end_line = len(lines)
    
    chunk_lines = lines[line_idx:end_line]
    chunk_text = '\n'.join(chunk_lines).strip()
    
    # Work out which source pages this chunk spans
    chunk_char_start = len('\n'.join(lines[:line_idx]))
    chunk_char_end = len('\n'.join(lines[:end_line]))
    
    pages_spanned = [
        p for p, offset in page_start_positions.items()
        if chunk_char_start <= offset <= chunk_char_end
    ]
    page_range = f"{min(pages_spanned)}–{max(pages_spanned)}" if pages_spanned else "unknown"
    
    chunks.append({
        "condition": condition,
        "category": category,
        "severity": severity,
        "source": "St John Ambulance First Aid Reference Guide",
        "page_range": page_range,
        "text": chunk_text,
    })

print(f"Created {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    word_count = len(c['text'].split())
    print(f"  [{i:>2}] {c['condition']:<45} | {c['category']:<25} | {c['severity']:<16} | ~{word_count} words")

Created 46 chunks

  [ 0] Assessing a casualty — Primary Survey         | Assessment                | core             | ~848 words
  [ 1] Secondary Survey                              | Assessment                | core             | ~157 words
  [ 2] Head-to-toe Survey                            | Assessment                | core             | ~476 words
  [ 3] Monitoring a Casualty                         | Assessment                | core             | ~209 words
  [ 4] Recovery Position                             | Unresponsive Casualty     | life-threatening | ~238 words
  [ 5] CPR — Adult, Child and Baby                   | Unresponsive Casualty     | life-threatening | ~1678 words
  [ 6] Automated External Defibrillation (AED)       | Unresponsive Casualty     | life-threatening | ~1705 words
  [ 7] CPR after Drowning                            | Unresponsive Casualty     | life-threatening | ~157 words
  [ 8] Seizures                                      | Response Problems   

In [13]:
# ── Cell 13: Inspect a specific chunk in full ──────────────────────────────
# Change this index to inspect any chunk
INSPECT = 5  # CPR chunk — good one to verify

c = chunks[INSPECT]
print(f"Condition : {c['condition']}")
print(f"Category  : {c['category']}")
print(f"Severity  : {c['severity']}")
print(f"Pages     : {c['page_range']}")
print(f"Words     : {len(c['text'].split())}")
print(f"\n{'─'*60}\n")
print(c['text'])

Condition : CPR — Adult, Child and Baby
Category  : Unresponsive Casualty
Severity  : life-threatening
Pages     : 23–28
Words     : 1678

────────────────────────────────────────────────────────────

Unresponsive not breathing casualty
Cardiopulmonary resuscitation (CPR)
Chest compressions and rescue breaths performed together are called
cardiopulmonary resuscitation (CPR). CPR is an important part of the chain of
survival for a casualty whose heart has stopped. The quality of CPR is dependent on
the correct depth and rate.
Chain of survival
- Recognising cardiac arrest
- Early request for help to the emergency services
- Early basic life support – CPR
- Early defibrillation
- Early advanced life support.
To provide the last two links in the chain you need to get help as soon as possible.
Defibrillation is known to greatly improve the outcome for some casualties whose
hearts have stopped. If a defibrillator is available, get it when calling the emergency
services: a defibrillator shou

In [14]:
# ── Cell 14: Improved page header/footer cleaning ─────────────────────────

def clean_page_text_v2(text: str) -> str:
    # Format 1: "First aid reference guide 15" (odd pages)
    text = re.sub(r'First aid reference guide\s+\d+\s*\n?', '', text)
    
    # Format 2: "24 First aid reference guide" (even pages)  
    text = re.sub(r'\d+\s+First aid reference guide\s*\n?', '', text)
    
    # Garbled lines — catch text that looks like random character soup
    # Heuristic: line with low ratio of real words (lots of mixed case clusters)
    def is_garbled(line: str) -> bool:
        if len(line) < 20:
            return False
        # Count transitions between upper and lower case — garbled text has many
        transitions = sum(
            1 for i in range(len(line) - 1)
            if line[i].isalpha() and line[i+1].isalpha()
            and line[i].isupper() != line[i+1].isupper()
        )
        # More than 1 transition per 3 chars is suspicious
        return transitions / len(line) > 0.33

    lines = text.split('\n')
    lines = [l for l in lines if not is_garbled(l)]
    text = '\n'.join(lines)

    # Replace `h` bullet artefact
    text = re.sub(r'^\s*h\s+', '- ', text, flags=re.MULTILINE)
    
    # Collapse excess blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


# Re-apply cleaning to all pages
pages_clean = {
    page_num: clean_page_text_v2(text)
    for page_num, text in pages_text.items()
}

# Verify the CPR chunk looks clean now — rebuild full text and re-chunk
full_text_with_markers = ""
page_start_positions = {}

for page_num in sorted(pages_clean.keys()):
    page_start_positions[page_num] = len(full_text_with_markers)
    full_text_with_markers += pages_clean[page_num] + "\n\n"

# Re-run the chunker from Cell 12
lines = full_text_with_markers.split('\n')
chunks = []

for i, (line_idx, condition, category, severity) in enumerate(CHUNK_BOUNDARIES):
    end_line = CHUNK_BOUNDARIES[i + 1][0] if i + 1 < len(CHUNK_BOUNDARIES) else len(lines)
    chunk_text = '\n'.join(lines[line_idx:end_line]).strip()
    
    chunk_char_start = len('\n'.join(lines[:line_idx]))
    chunk_char_end = len('\n'.join(lines[:end_line]))
    pages_spanned = [
        p for p, offset in page_start_positions.items()
        if chunk_char_start <= offset <= chunk_char_end
    ]
    page_range = f"{min(pages_spanned)}–{max(pages_spanned)}" if pages_spanned else "unknown"
    
    chunks.append({
        "condition": condition,
        "category": category,
        "severity": severity,
        "source": "St John Ambulance First Aid Reference Guide",
        "page_range": page_range,
        "text": chunk_text,
    })

print(f"Rebuilt {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    word_count = len(c['text'].split())
    print(f"  [{i:>2}] {c['condition']:<45} | ~{word_count:>5} words")

Rebuilt 46 chunks

  [ 0] Assessing a casualty — Primary Survey         | ~  852 words
  [ 1] Secondary Survey                              | ~  161 words
  [ 2] Head-to-toe Survey                            | ~  476 words
  [ 3] Monitoring a Casualty                         | ~  204 words
  [ 4] Recovery Position                             | ~  243 words
  [ 5] CPR — Adult, Child and Baby                   | ~ 1716 words
  [ 6] Automated External Defibrillation (AED)       | ~ 1687 words
  [ 7] CPR after Drowning                            | ~  168 words
  [ 8] Seizures                                      | ~  422 words
  [ 9] Febrile Convulsions                           | ~  251 words
  [10] Stroke                                        | ~  315 words
  [11] Head Injuries                                 | ~  357 words
  [12] Diabetes — Hypo and Hyperglycaemia            | ~  406 words
  [13] Choking                                       | ~  523 words
  [14] Asthma                

In [ ]:
# ── Cell 15 (revised): Install & configure Voyage ─────────────────────────

import voyageai
import chromadb

VOYAGE_API_KEY = "your-voyage-api-key-here"  # swap for env var in FastAPI
VOYAGE_MODEL = "voyage-4"

voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)

print(f"Voyage client ready, model: {VOYAGE_MODEL}")

Voyage client ready, model: voyage-4


In [16]:
# ── Cell 16 (revised): Build texts to embed ───────────────────────────────
# Prepend condition name to anchor the embedding to its topic

def build_embed_text(chunk: dict) -> str:
    return f"{chunk['condition']}\n\n{chunk['text']}"

texts_to_embed = [build_embed_text(c) for c in chunks]

print(f"Chunks to embed: {len(texts_to_embed)}")
print(f"\nSample (first 200 chars):")
print(texts_to_embed[0][:200])

Chunks to embed: 46

Sample (first 200 chars):
Assessing a casualty — Primary Survey

Assessing a casualty
When you encounter a casualty, you have to assess them so you can deliver
effective, safe and prompt first aid. There are two procedures you


In [17]:
# ── Cell 17 (revised): Generate embeddings via Voyage ─────────────────────
# Voyage batches up to 128 inputs per call — all 46 chunks fit in one call

print("Generating embeddings via Voyage AI...")

result = voyage_client.embed(
    texts_to_embed,
    model=VOYAGE_MODEL,
    input_type="document"   # "document" for chunks at index time
)

embeddings = result.embeddings
print(f"Done. {len(embeddings)} embeddings, dimension: {len(embeddings[0])}")
print(f"Total tokens used: {result.total_tokens}")

Generating embeddings via Voyage AI...
Done. 46 embeddings, dimension: 1024
Total tokens used: 22904


In [18]:
# ── Cell 18 (revised): Store in Chroma ────────────────────────────────────

chroma_client = chromadb.PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection("first_aid")
except:
    pass

collection = chroma_client.create_collection(
    name="first_aid",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings,
    documents=[c['text'] for c in chunks],
    metadatas=[{
        "condition":  c['condition'],
        "category":   c['category'],
        "severity":   c['severity'],
        "source":     c['source'],
        "page_range": c['page_range'],
    } for c in chunks]
)

print(f"Stored {collection.count()} chunks in Chroma collection 'first_aid'")

Stored 46 chunks in Chroma collection 'first_aid'


In [19]:
# ── Cell 19 (revised): Test retrieval ─────────────────────────────────────
# At query time, use input_type="query" — Voyage optimises differently
# for queries vs documents

def query_knowledge_base(query: str, n_results: int = 3):
    query_embedding = voyage_client.embed(
        [query],
        model=VOYAGE_MODEL,
        input_type="query"   # "query" for retrieval at request time
    ).embeddings[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )

    print(f"Query: '{query}'\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"  Result {i+1}: {meta['condition']}")
        print(f"    Category : {meta['category']}")
        print(f"    Severity : {meta['severity']}")
        print(f"    Distance : {dist:.4f}")
        print(f"    Preview  : {doc[:120]}...")
        print()

# Test queries
query_knowledge_base("someone is choking and can't breathe")
query_knowledge_base("heart has stopped not breathing")
query_knowledge_base("severe bleeding won't stop")
query_knowledge_base("person collapsed in the heat")
query_knowledge_base("what can I treat with ibuprofen and bandages")

Query: 'someone is choking and can't breathe'

  Result 1: Choking
    Category : Airway & Breathing
    Severity : life-threatening
    Distance : 0.3658
    Preview  : 2. Encourage the casualty to cough
3. If they cannot clear the object themselves, or they cannot cough or breathe,
suppo...

  Result 2: Diabetes — Hypo and Hyperglycaemia
    Category : Response Problems
    Severity : serious
    Distance : 0.3820
    Preview  : - Deep and sighing breathing
- Sweet smell on the breath
- Restless, drowsy or lethargic behaviour.
Treatment
Aims - To ...

  Result 3: Chemical Burns
    Category : Burns & Poisons
    Severity : serious
    Distance : 0.4565
    Preview  : 4. Dial 999 or 112 for an ambulance.
Smoke inhalation
- Redness, swelling or burning of the tongue
- Hoarse voice.

Trea...

Query: 'heart has stopped not breathing'

  Result 1: CPR — Adult, Child and Baby
    Category : Unresponsive Casualty
    Severity : life-threatening
    Distance : 0.4436
    Preview  : survival 